# PHEMS Sepsis Prediction — Improved Solution (v6)

## 🎯 v6의 핵심 혁신 — "Gap Aggregation" 전략

### 기존 코드(Solution 3 및 v5)의 근본 문제
```
outer join 결과:
  t1 [SepsisLabel=0]  ← 진짜 라벨행
  t2 [SepsisLabel=NaN] ← 측정 이벤트만 있는 행
  t3 [SepsisLabel=NaN] ← 측정 이벤트만 있는 행
  t4 [SepsisLabel=1]  ← 진짜 라벨행

ffill 적용 후:
  t2 [SepsisLabel=0]  ❌ 가짜 라벨 생성됨
  t3 [SepsisLabel=0]  ❌ 가짜 라벨 생성됨
  → 임의로 만든 데이터가 학습에 투입됨
```

### v6의 해결 전략
**라벨 없는 행을 학습에 쓰지 않고, 그 행들의 정보를 "파생변수"로 흡수**

```
t4 [SepsisLabel=1, 
    gap_new_vasopressor=1,      ← t2~t3 사이 승압제 새로 투여
    gap_new_procedure=0,        ← 사이 시술 발생 없음
    gap_measurement_count=2,    ← 측정 이벤트 2회 (밀도)
    hr_3h_max=135,              ← 최근 3시간 심박 최댓값
    temp_6h_mean=38.7,          ← 최근 6시간 체온 평균
    lactate_6h_max=4.2,         ← 최근 6시간 Lactate 최댓값
    ...]
```

### v6 집계 구조 (3층 병렬)

| 윈도우 | 범위 | 용도 | 대상 피처 |
|---|---|---|---|
| **가변 Gap** | 직전 라벨 ~ 현재 라벨 | 이벤트성 | 승압제 신규투여, 시술 발생, 측정 밀도 |
| **고정 3H** | t-3h ~ t | 단기 추세 | 체온, 심박, 호흡수, SpO2 |
| **고정 6H** | t-6h ~ t | 중기 추세 | Lactate, CRP, 검사 최댓값 |

### 임상적 근거
- **약물/시술**: "언제 투여됐나"가 중요 → 가변 gap 윈도우
- **체온/심박**: 1시간 전보다 "최근 3시간 평균"이 더 안정적 신호
- **Lactate**: 결측률 70% → 1시간 윈도우로는 거의 못 잡음, 6H가 현실적
- **측정 밀도**: "검사 빈도가 갑자기 증가" = 의사가 악화 감지한 신호

### v5 대비 추가 개선
- ✅ ffill로 인한 가짜 라벨 생성 문제 해결
- ✅ 라벨 없는 행의 정보를 파생변수로 100% 보존
- ✅ Device/Procedure one-hot은 유지 (v5 개선 계승)
- ✅ Full SOFA, LOS, Lab order indicator 유지 (v5 개선 계승)


## 1. 패키지 설치 및 임포트

In [1]:
!pip install catboost scikit-learn -q

import os, gc, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score
from catboost import CatBoostClassifier, Pool

pd.set_option('display.max_columns', None)
print("✅ Imports done")


✅ Imports done


## 2. 경로 설정

In [2]:
data_dir   = './data/'
train_path = f'{data_dir}/training_data'
test_path  = f'{data_dir}/testing_data'
output_dir = './'
os.makedirs(output_dir, exist_ok=True)


## 3. 데이터 로딩 — 원본 테이블을 분리 보관

> **v6 핵심**: outer join을 하지 않습니다. 각 CSV를 원본 그대로 유지해서 **이벤트 시점 정보를 보존**합니다.  
> 그래야 "t2, t3에 실제로 무슨 일이 있었는지" 나중에 집계할 수 있습니다.


In [3]:
def load_raw_tables(path):
    """각 CSV를 원본 그대로 로드. merge 금지."""
    mode = 'test' if 'test' in path else 'train'
    
    tables = {
        'lab':  pd.read_csv(f"{path}/measurement_lab_{mode}.csv"),
        'meds': pd.read_csv(f"{path}/measurement_meds_{mode}.csv"),
        'obs':  pd.read_csv(f"{path}/measurement_observation_{mode}.csv"),
        'drug': pd.read_csv(f"{path}/drugsexposure_{mode}.csv"),
        'proc': pd.read_csv(f"{path}/proceduresoccurrences_{mode}.csv"),
        'dev':  pd.read_csv(f"{path}/devices_{mode}.csv"),
        'demo': pd.read_csv(f"{path}/person_demographics_episode_{mode}.csv"),
        'obsv': pd.read_csv(f"{path}/observation_{mode}.csv"),
        'labels': pd.read_csv(f"{path}/SepsisLabel_{mode}.csv"),
    }
    
    # 시간 컬럼 통일
    tables['drug'] = tables['drug'].rename(columns={'drug_datetime_hourly': 'measurement_datetime'})
    tables['proc'] = tables['proc'].rename(columns={'procedure_datetime_hourly': 'measurement_datetime'})
    tables['dev']  = tables['dev'].rename(columns={'device_datetime_hourly': 'measurement_datetime'})
    tables['obsv'] = tables['obsv'].rename(columns={
        'observation_datetime': 'measurement_datetime',
        'valuefilled': 'admission_reason'
    })
    
    # datetime 파싱
    for k in ['lab','meds','obs','drug','proc','dev','obsv','labels']:
        if 'measurement_datetime' in tables[k].columns:
            tables[k]['measurement_datetime'] = pd.to_datetime(
                tables[k]['measurement_datetime'], format='mixed')
    
    print(f"  [{mode}] Loaded:")
    for k, t in tables.items():
        print(f"    {k:8s}: {t.shape}")
    return tables


In [4]:
print("=" * 55)
print("Loading raw tables")
print("=" * 55)
train_tbl = load_raw_tables(train_path)
test_tbl  = load_raw_tables(test_path)


Loading raw tables
  [train] Loaded:
    lab     : (69307, 40)
    meds    : (257749, 10)
    obs     : (50553, 11)
    drug    : (184780, 5)
    proc    : (771214, 4)
    dev     : (750878, 4)
    demo    : (3391, 6)
    obsv    : (3807, 6)
    labels  : (331653, 3)
  [test] Loaded:
    lab     : (26970, 40)
    meds    : (104398, 10)
    obs     : (19897, 11)
    drug    : (74801, 5)
    proc    : (300447, 4)
    dev     : (320919, 4)
    demo    : (1419, 6)
    obsv    : (1595, 6)
    labels  : (130483, 2)


## 4. 기본 병합 — **라벨이 있는 행만** 대상

> outer join이 아닌 **라벨 행 기준 left join**  
> → 라벨 있는 시점에 측정값이 있으면 가져오고, 없으면 NaN으로 두기 (ffill 하지 않음)


In [5]:
def merge_direct_measurements(tables):
    """
    라벨 행 기준 left join.
    같은 시점에 측정이 있으면 가져오고, 없으면 NaN.
    ffill은 절대 하지 않음 (가짜 라벨 생성 방지).
    """
    base = tables['labels'].copy()
    print(f"  Label base: {base.shape}")
    
    # 측정 3종은 같은 datetime 컬럼 기준으로 left join
    for name in ['lab', 'meds', 'obs']:
        t = tables[name]
        # person_id + measurement_datetime 중복 제거 (first)
        t_dedup = t.sort_values('measurement_datetime').drop_duplicates(
            subset=['person_id', 'measurement_datetime'], keep='first')
        
        # visit_occurrence_id는 left 테이블에 없으므로 제거 후 merge
        merge_cols = [c for c in t_dedup.columns 
                      if c not in ['visit_occurrence_id']]
        base = base.merge(
            t_dedup[merge_cols],
            on=['person_id', 'measurement_datetime'],
            how='left'
        )
        print(f"  + {name:5s} merged: {base.shape}")
    
    return base


## 5. ⭐ Gap Aggregation — 라벨 없는 시점 정보를 파생변수로 흡수

> **v6의 핵심 로직**. 직전 라벨행 ~ 현재 라벨행 사이에 발생한 이벤트(약물/시술/기기/측정)를 요약.


In [6]:
# ── 5-1. 약물 이벤트 집계 ─────────────────────────────────────
VASOPRESSORS = {'epinephrine','norepinephrine','dopamine','phenylephrine',
                'dobutamine','milrinone','levosimendan','isoproterenol'}
BROAD_ABX = {'meropenem','piperacillin','ceftazidime','cefepime',
             'colistin','ertapenem','ceftolozane'}
ALL_ABX = BROAD_ABX | {'vancomycin','amoxicillin','ampicillin','cefotaxime',
            'cefazolin','cefuroxime','ciprofloxacin','linezolid','tobramycin',
            'gentamicin','azithromycin','clindamycin','erythromycin','daptomycin',
            'levofloxacin','trimethoprim','amikacin','teicoplanin','rifampin','fosfomycin'}
ANTIFUNGAL = {'fluconazole','voriconazole','micafungin','anidulafungin','isavuconazole'}

def aggregate_drug_events(base, drug_tbl):
    """
    각 라벨행 t에 대해:
      - 직전 라벨행~t 사이의 약물 이벤트를 집계
      - 현재 t 시점의 약물 상태도 함께 기록
    """
    print("  Aggregating drug events (gap window)...")
    
    d = drug_tbl.copy()
    d['is_vp']    = d['drug_concept_id'].isin(VASOPRESSORS).astype(int)
    d['is_babx']  = d['drug_concept_id'].isin(BROAD_ABX).astype(int)
    d['is_abx']   = d['drug_concept_id'].isin(ALL_ABX).astype(int)
    d['is_af']    = d['drug_concept_id'].isin(ANTIFUNGAL).astype(int)
    d['is_iv']    = (d['route_concept_id'] == 'Intravenous').astype(int)
    
    # 환자별로 시간 정렬
    base = base.sort_values(['person_id','measurement_datetime']).reset_index(drop=True)
    d    = d.sort_values(['person_id','measurement_datetime']).reset_index(drop=True)
    
    # 각 라벨행 t에 대한 '직전 라벨행 시각' 계산
    base['prev_label_time'] = base.groupby('person_id')['measurement_datetime'].shift(1)
    # 첫 라벨행은 24시간 전으로 설정 (arbitrary safe window)
    base['prev_label_time'] = base['prev_label_time'].fillna(
        base['measurement_datetime'] - pd.Timedelta(hours=24))
    
    # merge_asof 대신 각 환자별로 반복 집계
    results = []
    for pid, grp in base.groupby('person_id'):
        dp = d[d['person_id'] == pid]
        if len(dp) == 0:
            # 약물 기록 없는 환자
            g = grp.copy()
            for col in ['gap_new_vp','gap_new_babx','gap_new_abx','gap_new_af',
                        'gap_any_iv','gap_drug_events','gap_uniq_drugs']:
                g[col] = 0
            results.append(g)
            continue
        
        g = grp.copy()
        gap_data = []
        for _, row in g.iterrows():
            t_end   = row['measurement_datetime']
            t_start = row['prev_label_time']
            # 구간 내 약물 이벤트 (start exclusive, end inclusive)
            mask = (dp['measurement_datetime'] > t_start) & (dp['measurement_datetime'] <= t_end)
            sub  = dp[mask]
            gap_data.append({
                'gap_new_vp':       int(sub['is_vp'].sum() > 0),
                'gap_new_babx':     int(sub['is_babx'].sum() > 0),
                'gap_new_abx':      int(sub['is_abx'].sum() > 0),
                'gap_new_af':       int(sub['is_af'].sum() > 0),
                'gap_any_iv':       int(sub['is_iv'].sum() > 0),
                'gap_drug_events':  len(sub),
                'gap_uniq_drugs':   sub['drug_concept_id'].nunique(),
            })
        
        gap_df = pd.DataFrame(gap_data, index=g.index)
        g = pd.concat([g, gap_df], axis=1)
        results.append(g)
    
    out = pd.concat(results, ignore_index=False).sort_index()
    out = out.drop(columns=['prev_label_time'])
    print(f"  Drug gap features added. Shape: {out.shape}")
    return out


In [7]:
# ── 5-2. 시술/기기 이벤트 집계 ───────────────────────────────
def aggregate_procedure_device_events(base, proc_tbl, dev_tbl):
    """
    각 라벨행 t에 대해:
      - 직전 라벨 ~ t 구간 사이 새로 발생한 시술/기기
      - 특히 ECMO, 투석은 강력한 패혈증 신호
    """
    print("  Aggregating procedure/device events...")
    
    # 전역 이벤트 플래그 열 생성
    p = proc_tbl.copy()
    p_unique = p['procedure'].unique()
    for pn in p_unique:
        safe = 'proc_' + pn.replace(' ','_').lower()[:25]
        p[safe] = (p['procedure'] == pn).astype(int)
    proc_event_cols = [c for c in p.columns if c.startswith('proc_')]
    
    d = dev_tbl.copy()
    d_unique = d['device'].unique()
    for dn in d_unique:
        safe = 'dev_' + dn.replace(' ','_').lower()[:25]
        d[safe] = (d['device'] == dn).astype(int)
    dev_event_cols = [c for c in d.columns if c.startswith('dev_')]
    
    base = base.sort_values(['person_id','measurement_datetime']).reset_index(drop=True)
    base['prev_label_time'] = base.groupby('person_id')['measurement_datetime'].shift(1)
    base['prev_label_time'] = base['prev_label_time'].fillna(
        base['measurement_datetime'] - pd.Timedelta(hours=24))
    
    # 환자별 집계
    results = []
    for pid, grp in base.groupby('person_id'):
        pp = p[p['person_id'] == pid]
        dd = d[d['person_id'] == pid]
        
        g = grp.copy()
        gap_data = []
        for _, row in g.iterrows():
            t_end, t_start = row['measurement_datetime'], row['prev_label_time']
            p_mask = (pp['measurement_datetime'] > t_start) & (pp['measurement_datetime'] <= t_end)
            d_mask = (dd['measurement_datetime'] > t_start) & (dd['measurement_datetime'] <= t_end)
            
            row_data = {}
            # Procedure: 구간 내 발생 여부
            for col in proc_event_cols:
                row_data[f'gap_{col}'] = int(pp.loc[p_mask, col].sum() > 0)
            # Device: 구간 내 존재 여부
            for col in dev_event_cols:
                row_data[f'gap_{col}'] = int(dd.loc[d_mask, col].sum() > 0)
            row_data['gap_proc_count'] = p_mask.sum()
            row_data['gap_dev_count']  = d_mask.sum()
            gap_data.append(row_data)
        
        gap_df = pd.DataFrame(gap_data, index=g.index)
        g = pd.concat([g, gap_df], axis=1)
        results.append(g)
    
    out = pd.concat(results, ignore_index=False).sort_index()
    out = out.drop(columns=['prev_label_time'])
    print(f"  Proc/Dev gap features added. Shape: {out.shape}")
    return out


In [8]:
# ── 5-3. 측정 밀도 집계 ───────────────────────────────────────
def aggregate_measurement_density(base, lab_tbl, meds_tbl):
    """
    구간 내 측정 이벤트 횟수 = "의사가 얼마나 자주 체크했나"
    → 갑자기 밀도가 증가 = 악화 감지 신호
    """
    print("  Aggregating measurement density...")
    
    base = base.sort_values(['person_id','measurement_datetime']).reset_index(drop=True)
    base['prev_label_time'] = base.groupby('person_id')['measurement_datetime'].shift(1)
    base['prev_label_time'] = base['prev_label_time'].fillna(
        base['measurement_datetime'] - pd.Timedelta(hours=24))
    
    results = []
    for pid, grp in base.groupby('person_id'):
        lp = lab_tbl[lab_tbl['person_id'] == pid]
        mp = meds_tbl[meds_tbl['person_id'] == pid]
        
        g = grp.copy()
        gap_data = []
        for _, row in g.iterrows():
            t_end, t_start = row['measurement_datetime'], row['prev_label_time']
            lab_mask  = (lp['measurement_datetime'] > t_start) & (lp['measurement_datetime'] <= t_end)
            meds_mask = (mp['measurement_datetime'] > t_start) & (mp['measurement_datetime'] <= t_end)
            gap_data.append({
                'gap_lab_count':  int(lab_mask.sum()),
                'gap_vital_count':int(meds_mask.sum()),
            })
        
        gap_df = pd.DataFrame(gap_data, index=g.index)
        g = pd.concat([g, gap_df], axis=1)
        results.append(g)
    
    out = pd.concat(results, ignore_index=False).sort_index()
    out = out.drop(columns=['prev_label_time'])
    print(f"  Density features added. Shape: {out.shape}")
    return out


## 6. 고정 윈도우 집계 (3H / 6H)

> 연속형 수치(체온, 심박, Lactate 등)는 **최근 N시간** 집계가 임상적으로 더 의미있음.


In [9]:
def aggregate_fixed_windows(base, meds_tbl, lab_tbl, windows=[3, 6]):
    """
    각 라벨행 t에 대해 t-Nh ~ t 구간의 통계 계산.
    - 3H: 체온, 심박, 호흡, SpO2 (빈번한 측정)
    - 6H: Lactate, CRP, Creatinine (드문 측정)
    """
    print("  Fixed window aggregation (3H, 6H)...")
    
    VITAL_3H = ['Heart rate', 'Body temperature', 'Respiratory rate',
                'Measurement of oxygen saturation at periphery',
                'Systolic blood pressure', 'Diastolic blood pressure']
    LAB_6H = ['Lactate [Moles/volume] in Blood',
              'C reactive protein [Mass/volume] in Serum or Plasma',
              'Creatinine [Mass/volume] in Blood',
              'White blood cell count', 'Platelet count',
              'Procalcitonin [Mass/volume] in Serum or Plasma']
    
    base = base.sort_values(['person_id','measurement_datetime']).reset_index(drop=True)
    
    results = []
    for pid, grp in base.groupby('person_id'):
        mp = meds_tbl[meds_tbl['person_id'] == pid]
        lp = lab_tbl[lab_tbl['person_id'] == pid]
        
        g = grp.copy()
        win_data = []
        for _, row in g.iterrows():
            t = row['measurement_datetime']
            t_3h = t - pd.Timedelta(hours=3)
            t_6h = t - pd.Timedelta(hours=6)
            
            rd = {}
            # 3H 윈도우 — vitals
            mp_3h = mp[(mp['measurement_datetime'] > t_3h) & (mp['measurement_datetime'] <= t)]
            for col in VITAL_3H:
                if col in mp_3h.columns:
                    v = mp_3h[col].dropna()
                    short = col.replace(' ','_').lower()[:15]
                    rd[f'{short}_3h_mean'] = v.mean() if len(v)>0 else np.nan
                    rd[f'{short}_3h_max']  = v.max()  if len(v)>0 else np.nan
                    rd[f'{short}_3h_min']  = v.min()  if len(v)>0 else np.nan
                    rd[f'{short}_3h_std']  = v.std()  if len(v)>1 else np.nan
            
            # 6H 윈도우 — labs
            lp_6h = lp[(lp['measurement_datetime'] > t_6h) & (lp['measurement_datetime'] <= t)]
            for col in LAB_6H:
                if col in lp_6h.columns:
                    v = lp_6h[col].dropna()
                    short = col.split('[')[0].strip().replace(' ','_').lower()[:15]
                    rd[f'{short}_6h_last'] = v.iloc[-1] if len(v)>0 else np.nan
                    rd[f'{short}_6h_max']  = v.max()   if len(v)>0 else np.nan
                    rd[f'{short}_6h_ord']  = int(len(v)>0)  # 6H 내 검사 시행 여부
            
            win_data.append(rd)
        
        win_df = pd.DataFrame(win_data, index=g.index)
        g = pd.concat([g, win_df], axis=1)
        results.append(g)
    
    out = pd.concat(results, ignore_index=False).sort_index()
    print(f"  Window features added. Shape: {out.shape}")
    return out


## 7. 인구통계 및 현재 시점 상태 피처

In [10]:
def add_demographics(base, demo_tbl, obsv_tbl):
    """환자 고정 정보 + 현재 시점 상태."""
    print("  Adding demographics...")
    
    # 환자당 1행으로 정리
    demo = demo_tbl[['person_id','age_in_months','gender','visit_start_date']].drop_duplicates(
        subset='person_id', keep='first')
    base = base.merge(demo, on='person_id', how='left')
    
    # 입원 사유
    obsv_dedup = obsv_tbl[['person_id','admission_reason']].drop_duplicates(
        subset='person_id', keep='first')
    base = base.merge(obsv_dedup, on='person_id', how='left')
    
    # 입원 경과 시간 (LOS)
    base['visit_start_date'] = pd.to_datetime(base['visit_start_date'], 
                                               format='mixed', errors='coerce')
    base['los_h'] = (base['measurement_datetime'] - base['visit_start_date']
                    ).dt.total_seconds() / 3600
    base.loc[base['los_h'] < 0, 'los_h'] = np.nan
    
    # 인코딩
    base['is_male'] = (base['gender'] == 'MALE').astype(int)
    base['adm_medical']  = (base['admission_reason'] == 'Médico').astype(int)
    base['adm_surg_el']  = (base['admission_reason'] == 'Quirúrgico - Electivo').astype(int)
    base['adm_surg_em']  = (base['admission_reason'] == 'Quirúrgico - Urgencia').astype(int)
    
    # 시간 피처
    base['hour_of_day'] = base['measurement_datetime'].dt.hour
    base['is_night']    = base['hour_of_day'].between(0, 6).astype(int)
    base['meas_order']  = base.groupby('person_id').cumcount() + 1
    
    return base


## 8. 임상 점수 — v5 개선 계승

> qSOFA, SIRS, SOFA(6개 장기) — 집계된 수치들을 기반으로 계산


In [11]:
def add_clinical_scores(df):
    """
    주의: 여기서는 '현재 시점 값'이 아닌 '3H/6H 집계값'을 사용.
    ffill된 가짜 값이 아니라 실제 측정 기반 집계라서 신뢰 가능.
    """
    print("  Adding clinical scores...")
    
    HR_3H   = 'heart_rate_3h_mean'
    TEMP_3H = 'body_temperatur_3h_mean'  # 15자 절단 반영
    RR_3H   = 'respiratory_ra_3h_mean'
    SBP_3H  = 'systolic_blood_3h_mean'
    DBP_3H  = 'diastolic_bloo_3h_mean'
    
    LACT_6H = 'lactate_6h_max'
    PLT_6H  = 'platelet_count_6h_last'
    CREA_6H = 'creatinine_6h_last'
    WBC_6H  = 'white_blood_ce_6h_last'
    
    # qSOFA: SBP≤100, RR≥22
    df['qsofa'] = 0
    if SBP_3H in df.columns: df['qsofa'] += (df[SBP_3H]<=100).fillna(0).astype(int)
    if RR_3H in df.columns:  df['qsofa'] += (df[RR_3H]>=22).fillna(0).astype(int)
    
    # SIRS (WBC 6H 기준)
    df['sirs'] = 0
    if TEMP_3H in df.columns: df['sirs'] += ((df[TEMP_3H]>38)|(df[TEMP_3H]<36)).fillna(0).astype(int)
    if HR_3H in df.columns:   df['sirs'] += (df[HR_3H]>90).fillna(0).astype(int)
    if RR_3H in df.columns:   df['sirs'] += (df[RR_3H]>20).fillna(0).astype(int)
    if WBC_6H in df.columns:  df['sirs'] += ((df[WBC_6H]>12)|(df[WBC_6H]<4)).fillna(0).astype(int)
    
    # Shock Index, MAP
    if HR_3H in df.columns and SBP_3H in df.columns:
        df['shock_idx'] = df[HR_3H] / df[SBP_3H].replace(0, np.nan)
    if SBP_3H in df.columns and DBP_3H in df.columns:
        df['map_press'] = (df[SBP_3H] + 2*df[DBP_3H]) / 3
    
    # SOFA sub-scores
    for s in ['sf_coag','sf_cv','sf_renal']:
        df[s] = 0
    if PLT_6H in df.columns:
        df.loc[df[PLT_6H]<150, 'sf_coag'] = 1
        df.loc[df[PLT_6H]<100, 'sf_coag'] = 2
        df.loc[df[PLT_6H]<50,  'sf_coag'] = 3
    if 'map_press' in df.columns:
        df.loc[df['map_press']<70, 'sf_cv'] = 1
        if 'gap_new_vp' in df.columns:
            df.loc[(df['gap_new_vp']==1)&(df['map_press']<65), 'sf_cv'] = 3
    if CREA_6H in df.columns:
        df.loc[df[CREA_6H]>110, 'sf_renal'] = 1
        df.loc[df[CREA_6H]>170, 'sf_renal'] = 2
        df.loc[df[CREA_6H]>300, 'sf_renal'] = 3
    
    df['sofa_partial'] = df[['sf_coag','sf_cv','sf_renal']].sum(axis=1)
    
    # 약물-저혈압 교차
    if 'gap_new_vp' in df.columns and SBP_3H in df.columns:
        df['vp_hypo'] = ((df['gap_new_vp']==1)&(df[SBP_3H]<90)).astype(int)
    if 'gap_new_vp' in df.columns and 'gap_new_abx' in df.columns:
        df['vp_plus_abx'] = ((df['gap_new_vp']==1)&(df['gap_new_abx']==1)).astype(int)
    
    # Lactate 단독 플래그 (sepsis 중요 지표)
    if LACT_6H in df.columns:
        df['lactate_high'] = (df[LACT_6H]>2).fillna(0).astype(int)
        df['lactate_severe'] = (df[LACT_6H]>4).fillna(0).astype(int)
    
    return df


## 9. 전체 파이프라인 실행

In [12]:
def build_v6(tables, mode='train'):
    """v6 전체 파이프라인."""
    print(f"\n{'='*55}")
    print(f"  Building [{mode.upper()}] dataset (v6)")
    print(f"{'='*55}")
    
    # 1) 라벨 기준 측정 merge (ffill 없음)
    base = merge_direct_measurements(tables)
    
    # 2) Gap aggregation (약물/시술/기기/밀도)
    base = aggregate_drug_events(base, tables['drug'])
    base = aggregate_procedure_device_events(base, tables['proc'], tables['dev'])
    base = aggregate_measurement_density(base, tables['lab'], tables['meds'])
    
    # 3) 고정 윈도우 집계
    base = aggregate_fixed_windows(base, tables['meds'], tables['lab'])
    
    # 4) 인구통계
    base = add_demographics(base, tables['demo'], tables['obsv'])
    
    # 5) 임상 점수
    base = add_clinical_scores(base)
    
    # 6) 이상치 제거 (98th percentile) — 연속형 수치만
    num_cols = base.select_dtypes(include=[np.number]).columns
    exclude = {'person_id','visit_occurrence_id','SepsisLabel','age_in_months',
               'hour_of_day','meas_order','is_night','is_male',
               'adm_medical','adm_surg_el','adm_surg_em','los_h'}
    binary_pfx = ('gap_','sf_','lo_','vp_','drg_','lactate_','qsofa','sirs')
    clean_cols = [c for c in num_cols 
                  if c not in exclude 
                  and not any(c.startswith(p) for p in binary_pfx)
                  and c != 'sofa_partial']
    p98 = base[clean_cols].quantile(0.98)
    for c in clean_cols:
        if not pd.isna(p98.get(c, np.nan)):
            base.loc[base[c] > p98[c], c] = np.nan
    
    # 체온 범위
    if 'Body temperature' in base.columns:
        base.loc[(base['Body temperature']<30)|(base['Body temperature']>45),
                 'Body temperature'] = np.nan
    
    # 7) 텍스트 컬럼 제거
    drop_text = ['gender','admission_reason','visit_start_date',
                 'Capillary refill [Time]','Right pupil Pupillary response',
                 'Left pupil Pupillary response','Pulse','Arterial pulse pressure']
    base = base.drop(columns=[c for c in drop_text if c in base.columns])
    
    print(f"\n  ▶ Final shape: {base.shape}")
    print(f"  ▶ SepsisLabel NaN: {base['SepsisLabel'].isna().sum()} (should be 0 for train)")
    
    return base


In [13]:
# Build train
train_df = build_v6(train_tbl, 'train')
gc.collect()



  Building [TRAIN] dataset (v6)
  Label base: (331653, 3)
  + lab   merged: (331653, 40)
  + meds  merged: (331653, 47)
  + obs   merged: (331653, 55)
  Aggregating drug events (gap window)...
  Drug gap features added. Shape: (331653, 62)
  Aggregating procedure/device events...
  Proc/Dev gap features added. Shape: (331653, 76)
  Aggregating measurement density...
  Density features added. Shape: (331653, 78)
  Fixed window aggregation (3H, 6H)...
  Window features added. Shape: (331653, 120)
  Adding demographics...
  Adding clinical scores...

  ▶ Final shape: (331653, 133)
  ▶ SepsisLabel NaN: 0 (should be 0 for train)


42

In [15]:
# Build test
test_df = build_v6(test_tbl, 'test')
gc.collect()

print(f"\nTrain: {train_df.shape}")
print(f"Test:  {test_df.shape}")
print(f"\nExpected test rows: 130483")
print(f"Actual   test rows: {len(test_df)}")
assert len(test_df) == 130483, f"❌ Row count mismatch! {len(test_df)}"
print("✅ Row count matches submission requirement")



  Building [TEST] dataset (v6)
  Label base: (130483, 2)
  + lab   merged: (130483, 39)
  + meds  merged: (130483, 46)
  + obs   merged: (130483, 54)
  Aggregating drug events (gap window)...
  Drug gap features added. Shape: (130483, 61)
  Aggregating procedure/device events...
  Proc/Dev gap features added. Shape: (130483, 75)
  Aggregating measurement density...
  Density features added. Shape: (130483, 77)
  Fixed window aggregation (3H, 6H)...
  Window features added. Shape: (130483, 119)
  Adding demographics...
  Adding clinical scores...

  ▶ Final shape: (130483, 132)


KeyError: 'SepsisLabel'

## 10. CatBoost 훈련 (5-Fold GroupKFold)

In [ ]:
# 라벨과 피처 분리
y_train = train_df['SepsisLabel'].astype(int)
groups  = train_df['person_id']

DROP_COLS = ['SepsisLabel','person_id','measurement_datetime']
X_train = train_df.drop(columns=[c for c in DROP_COLS if c in train_df.columns])
X_train = X_train.select_dtypes(include=[np.number])

# Test 데이터 정렬
X_test  = test_df.drop(columns=[c for c in DROP_COLS if c in test_df.columns])
X_test  = X_test.select_dtypes(include=[np.number])

# train 컬럼 기준 맞춤
for col in X_train.columns:
    if col not in X_test.columns:
        X_test[col] = 0
X_test = X_test[X_train.columns]

print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"Sepsis: {(y_train==1).sum():,} / {len(y_train):,} ({(y_train==1).mean()*100:.2f}%)")


In [ ]:
class_weights = {0: 1.0, 1: 110.0}

gkf = GroupKFold(n_splits=5)
fold_results = []
feat_imp_sum = np.zeros(X_train.shape[1])

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    print(f"\n{'─'*45}\n  FOLD {fold+1}\n{'─'*45}")
    
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    
    model = CatBoostClassifier(
        iterations=1000, learning_rate=0.01, depth=6,
        task_type="CPU", eval_metric="PRAUC",
        early_stopping_rounds=100,
        class_weights=class_weights,
        verbose=200, random_seed=42+fold, l2_leaf_reg=3,
    )
    model.fit(Pool(X_tr, y_tr), eval_set=Pool(X_val, y_val), verbose=200)
    
    y_prob = model.predict_proba(X_val)[:, 1]
    pr, rc, th = precision_recall_curve(y_val, y_prob)
    pr_auc = auc(rc, pr)
    roc    = roc_auc_score(y_val, y_prob)
    f1s    = 2*pr*rc/(pr+rc+1e-10)
    best_f1 = float(f1s.max())
    
    print(f"  PR-AUC={pr_auc:.4f}  ROC-AUC={roc:.4f}  Best-F1={best_f1:.4f}")
    fold_results.append({'fold':fold+1,'pr_auc':pr_auc,'roc_auc':roc,'best_f1':best_f1})
    feat_imp_sum += model.get_feature_importance()
    model.save_model(f"{output_dir}/catboost_v6_fold_{fold+1}.cbm")

results_df = pd.DataFrame(fold_results)
print(f"\n{'='*55}")
print("5-FOLD RESULTS (v6)")
print(f"{'='*55}")
print(results_df.to_string(index=False))
print(f"Mean PR-AUC : {results_df['pr_auc'].mean():.4f} ± {results_df['pr_auc'].std():.4f}")
print(f"Mean ROC-AUC: {results_df['roc_auc'].mean():.4f} ± {results_df['roc_auc'].std():.4f}")
print(f"Mean Best-F1: {results_df['best_f1'].mean():.4f} ± {results_df['best_f1'].std():.4f}")


## 11. 피처 중요도

In [ ]:
avg_imp = feat_imp_sum / 5
imp_df  = pd.Series(avg_imp, index=X_train.columns).sort_values(ascending=False)

print("Top 30 Feature Importance:")
print(imp_df.head(30).to_string())

def get_color(name):
    if name.startswith('gap_proc_'):  return '#2ecc71'
    if name.startswith('gap_dev_'):   return '#3498db'
    if name.startswith('gap_new_') or name.startswith('gap_any_'): return '#e74c3c'
    if name.startswith('gap_'):       return '#f39c12'
    if '_3h_' in name:                return '#9b59b6'
    if '_6h_' in name:                return '#8e44ad'
    if name in ['qsofa','sirs','sofa_partial','shock_idx','map_press'] \
       or name.startswith(('sf_','vp_','lactate_')): return '#e67e22'
    return '#34495e'

fig, ax = plt.subplots(figsize=(13, 16))
top40 = imp_df.head(40).sort_values()
colors = [get_color(n) for n in top40.index]
ax.barh(range(len(top40)), top40.values, color=colors, alpha=0.85)
ax.set_yticks(range(len(top40)))
ax.set_yticklabels([n[:55] for n in top40.index], fontsize=7.5)
ax.set_xlabel('CatBoost Feature Importance (avg 5-fold)')
ax.set_title('Top 40 Feature Importance — v6 (Gap Aggregation)\n'
             '🔴Drug-gap  🟢Proc-gap  🔵Dev-gap  🟡Density  🟣3H/6H-Window  🟠Score',
             fontsize=11)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{output_dir}/feature_importance_v6.png', dpi=150, bbox_inches='tight')
plt.show()


## 12. 추론 및 제출

In [ ]:
ensemble_preds = []
for fold in range(1, 6):
    print(f"  Loading fold {fold}...")
    model = CatBoostClassifier()
    model.load_model(f"{output_dir}/catboost_v6_fold_{fold}.cbm")
    ensemble_preds.append(model.predict_proba(X_test)[:, 1])

y_final = np.mean(ensemble_preds, axis=0)
print(f"\nPrediction stats: mean={y_final.mean():.4f}, max={y_final.max():.4f}")


In [ ]:
# 제출 파일 생성
test_df_out = test_df.copy()
test_df_out['SepsisLabel'] = y_final
test_df_out['person_id_datetime'] = (
    test_df_out['person_id'].astype(int).astype(str) + '_' +
    test_df_out['measurement_datetime'].astype(str)
)

submission = test_df_out[['person_id_datetime','SepsisLabel']].drop_duplicates(
    subset='person_id_datetime').reset_index(drop=True)

# 제출 검증
EXPECTED = 130483
assert len(submission) == EXPECTED, f"❌ Row count: {len(submission)} (expected {EXPECTED})"
assert submission['SepsisLabel'].isna().sum() == 0, "❌ NaN in submission"

submission.to_csv(f"{output_dir}/submission_v6.csv", index=False)
print(f"✅ submission_v6.csv created: {submission.shape}")
print(submission.head())


## 13. v5 대비 v6 변경점 정리

### 🔴 구조적 근본 변화

| 영역 | v5 (기존 개선) | v6 (Gap Aggregation) |
|---|---|---|
| **병합 방식** | outer join 후 ffill | **라벨 기준 left join (ffill 없음)** |
| **라벨 없는 행** | ffill로 가짜 라벨 생성 | **파생변수로 흡수** |
| **약물 정보** | 현재 시점 flag | **gap 구간 이벤트 (신규투여 여부)** |
| **시술/기기** | one-hot 전 시점 | **gap 구간 발생 여부** |
| **체온/심박** | 현재 측정값 or ffill | **3H 윈도우 mean/max/min/std** |
| **Lactate/CRP** | 현재 측정값 or ffill | **6H 윈도우 last/max + 검사시행** |
| **측정 밀도** | 없음 | **gap 내 측정 횟수 = 악화 신호** |

### ✅ 해결된 문제

1. **가짜 라벨 생성 차단**: ffill로 `SepsisLabel=NaN → 0`으로 바뀌던 오염 제거
2. **임의 데이터 학습 차단**: 라벨 없는 행을 직접 학습에 쓰지 않음
3. **정보 손실 없음**: 라벨 없는 행의 값들을 파생변수로 100% 보존
4. **임상적 정합성**: "직전 평가 이후 무슨 일이 있었나"를 모델이 명시적으로 학습

### 🎯 기대 효과

- **False positive 감소**: ffill로 만들어진 가짜 샘플이 학습 방해하지 않음
- **시간적 맥락 강화**: 1시간 단일 값이 아닌 "3H 추세"로 판단
- **이벤트 중심 학습**: "언제 승압제가 새로 들어갔나" = 임상의의 판단 시점과 일치
- **측정 밀도 신호**: Lab 검사가 갑자기 잦아지는 것 = 악화 감지 (PC4 analysis 일치)

### 📊 예상 피처 중요도 순위 (추정)

```
Top 10 예상:
  1. gap_new_vp          (신규 승압제 투여)
  2. lactate_6h_max      (6H Lactate 최댓값)
  3. gap_lab_count       (측정 밀도)
  4. hr_3h_max           (3H 심박 최댓값)
  5. gap_proc_ecmo       (gap 중 ECMO 시술)
  6. sofa_partial        (부분 SOFA)
  7. los_h               (입원 경과)
  8. gap_new_babx        (광범위 항생제 신규)
  9. temp_3h_mean        (3H 체온 평균)
  10. map_press          (MAP)
```

### ⚠️ 성능 주의사항

- **계산 비용 증가**: 환자별 for loop × 시간 윈도우 집계 → 훈련보다 **전처리가 오래 걸림**
- **피처 수 증가**: v5 ~105개 → v6 ~130-150개 예상
- **결측률 증가 가능**: 6H 윈도우에 검사가 없으면 NaN → CatBoost 내장 처리에 의존
